# BERT — SGD · Test Multi-Seeds (5 seeds)
**Objectif :** Confirmer que l'échec de SGD est systématique — pas dû au hasard d'une seule initialisation.  
**Dataset :** SST-2 (GLUE) · **Modèle :** bert-base-uncased · **Seeds :** 42, 123, 2024, 7, 999  

> **Ce notebook doit être exécuté APRÈS `adamw-multiseed-test.ipynb`** pour pouvoir comparer les deux optimiseurs avec le test statistique de Welch.

In [ ]:
# Cell 0 — Installation des dépendances
!pip install transformers datasets torch scikit-learn scipy -q

In [ ]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import random
import time
from scipy import stats
from sklearn.metrics import f1_score, accuracy_score

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
    set_seed
)
import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')

In [ ]:
# Cell 2 — Configuration
MODEL_NAME   = 'bert-base-uncased'
SEEDS        = [42, 123, 2024, 7, 999]  # MÊMES seeds qu'AdamW
LR           = 2e-5
BATCH_SIZE   = 16
NUM_EPOCHS   = 3  # SGD a besoin d'une époque de plus
MAX_LENGTH   = 128
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MOMENTUM     = 0.9
GRAD_CLIP    = 1.0

print('Configuration SGD — Multi-Seeds')
print(f'  Seeds        : {SEEDS}')
print(f'  lr           : {LR}')
print(f'  momentum     : {MOMENTUM}')
print(f'  nesterov     : True')
print(f'  weight_decay : {WEIGHT_DECAY}')
print(f'  epochs       : {NUM_EPOCHS} (1 de plus que AdamW)')
print(f'  Runs totales : {len(SEEDS)}')

In [ ]:
# Cell 3 — Dataset + Tokenisation
dataset   = load_dataset('glue', 'sst2')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples['sentence'], truncation=True, max_length=MAX_LENGTH)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.remove_columns(['sentence', 'idx'])
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch')

collator      = DataCollatorWithPadding(tokenizer)
train_dataset = tokenized['train']
val_dataset   = tokenized['validation']

val_loader = DataLoader(val_dataset, batch_size=32, collate_fn=collator)

print(f'Train : {len(train_dataset)} exemples')
print(f'Val   : {len(val_dataset)} exemples')

In [ ]:
# Cell 4 — Fonctions utilitaires
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch  = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds  = logits.argmax(dim=-1).cpu().tolist()
            labels = batch['labels'].cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels)
    acc = accuracy_score(all_labels, all_preds) * 100
    f1  = f1_score(all_labels, all_preds, average='weighted') * 100
    return acc, f1

print('Fonctions prêtes.')

In [ ]:
# Cell 5 — Boucle Multi-Seeds SGD

# Config redéfinie ici pour éviter les erreurs si une cellule est sautée
MODEL_NAME   = 'bert-base-uncased'
SEEDS        = [42, 123, 2024, 7, 999]   # MÊMES seeds qu'AdamW
LR           = 2e-5
BATCH_SIZE   = 16
NUM_EPOCHS   = 3                          # SGD a besoin d'une époque de plus
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MOMENTUM     = 0.9
GRAD_CLIP    = 1.0
SEP          = '=' * 50

results = []  # [{seed, acc, f1, time_s}]

for seed in SEEDS:
    print('\n' + SEP)
    print('Seed ' + str(seed) + ' — SGD + Nesterov')
    print(SEP)
    set_all_seeds(seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(device)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE,
        shuffle=True, collate_fn=collator
    )

    optimizer = optim.SGD(
        model.parameters(),
        lr=LR,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
        nesterov=True
    )

    num_steps  = len(train_loader) * NUM_EPOCHS
    num_warmup = int(num_steps * WARMUP_RATIO)
    scheduler  = get_linear_schedule_with_warmup(
        optimizer, num_warmup, num_steps
    )

    t0 = time.time()
    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        for step, batch in enumerate(train_loader):
            batch  = {k: v.to(device) for k, v in batch.items()}
            loss   = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_acc, val_f1 = evaluate(model, val_loader)
        elapsed = time.time() - t0
        print('  Epoch ' + str(epoch+1) + '/' + str(NUM_EPOCHS) +
              ' | loss=' + str(round(avg_loss, 4)) +
              ' | val_acc=' + str(round(val_acc, 2)) + '%' +
              ' | f1=' + str(round(val_f1, 2)) + '%' +
              ' | ' + str(int(elapsed)) + 's')

    total_time = time.time() - t0
    final_acc, final_f1 = evaluate(model, val_loader)
    results.append({'seed': seed, 'acc': final_acc, 'f1': final_f1, 'time_s': total_time})
    print('\n  Résultat final seed=' + str(seed) +
          ' → acc=' + str(round(final_acc, 2)) + '%' +
          ' | f1=' + str(round(final_f1, 2)) + '%' +
          ' | temps=' + str(int(total_time)) + 's')

print('\nToutes les seeds terminées !')


In [ ]:
# Cell 6 — Statistiques SGD
accs  = [r['acc']    for r in results]
f1s   = [r['f1']     for r in results]
times = [r['time_s'] for r in results]

def bootstrap_ci(data, n_boot=2000, ci=0.95):
    boots = [np.mean(np.random.choice(data, len(data), replace=True)) for _ in range(n_boot)]
    return np.percentile(boots, [((1-ci)/2)*100, (1-(1-ci)/2)*100])

ci = bootstrap_ci(accs)

print('='*55)
print('RÉSULTATS STATISTIQUES — SGD · BERT · SST-2')
print('='*55)
print(f'  Runs          : {len(SEEDS)} seeds')
print(f'  Accuracy mean : {np.mean(accs):.2f}%')
print(f'  Accuracy std  : ±{np.std(accs, ddof=1):.2f}%')
print(f'  Accuracy min  : {np.min(accs):.2f}%')
print(f'  Accuracy max  : {np.max(accs):.2f}%')
print(f'  IC 95%        : [{ci[0]:.2f}%, {ci[1]:.2f}%]')
print(f'  F1 mean       : {np.mean(f1s):.2f}% ±{np.std(f1s, ddof=1):.2f}')
print(f'  Temps moyen   : {np.mean(times)/60:.1f} min/run')
print()

df = pd.DataFrame(results)
df['optimiseur'] = 'SGD'
df.to_csv('sgd_multiseed_results.csv', index=False)
print('Résultats sauvegardés : sgd_multiseed_results.csv')
print(df[['seed','acc','f1','time_s']].to_string(index=False))

In [ ]:
# Cell 7 — Test de Welch + Cohen's d (comparaison avec AdamW)
import os

sgd_accs = accs

# Charger les résultats AdamW si disponibles
if os.path.exists('adamw_multiseed_results.csv'):
    df_adamw   = pd.read_csv('adamw_multiseed_results.csv')
    adamw_accs = df_adamw['acc'].tolist()

    # Test de Welch (variances inégales)
    t_stat, p_val = stats.ttest_ind(adamw_accs, sgd_accs, equal_var=False)

    # Cohen's d (taille d'effet)
    pooled_std = np.sqrt((np.std(adamw_accs, ddof=1)**2 + np.std(sgd_accs, ddof=1)**2) / 2)
    cohens_d   = (np.mean(adamw_accs) - np.mean(sgd_accs)) / pooled_std

    print('='*60)
    print('TEST STATISTIQUE — AdamW vs SGD · Welch t-test')
    print('='*60)
    print(f'  AdamW : {np.mean(adamw_accs):.2f}% ± {np.std(adamw_accs, ddof=1):.2f}')
    print(f'  SGD   : {np.mean(sgd_accs):.2f}% ± {np.std(sgd_accs, ddof=1):.2f}')
    print(f'  Écart : +{np.mean(adamw_accs) - np.mean(sgd_accs):.2f}% en faveur d\'AdamW')
    print()
    print(f'  t-statistic : {t_stat:.4f}')
    print(f'  p-value     : {p_val:.6f}')
    print(f'  Cohen\'s d   : {cohens_d:.2f}')
    print()
    if p_val < 0.001:
        print('  Conclusion : *** p < 0.001 — différence HAUTEMENT significative')
    elif p_val < 0.01:
        print('  Conclusion : ** p < 0.01 — différence très significative')
    elif p_val < 0.05:
        print('  Conclusion : * p < 0.05 — différence significative')
    else:
        print('  Conclusion : ns — différence non significative')

    effect = 'négligeable' if abs(cohens_d) < 0.2 else ('petit' if abs(cohens_d) < 0.5 else ('moyen' if abs(cohens_d) < 0.8 else 'large'))
    print(f'  Taille effet: {effect} (d={cohens_d:.2f})')
else:
    print('Fichier adamw_multiseed_results.csv non trouvé.')
    print('Exécutez d\'abord adamw-multiseed-test.ipynb pour le test comparatif.')

In [ ]:
# Cell 8 — Visualisation comparative AdamW vs SGD
import os

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('AdamW vs SGD — BERT — SST-2 · Validation Multi-Seeds (n=5)', fontsize=13, fontweight='bold')

# Plot 1 : barplot comparatif
ax = axes[0]
opt_names, opt_means, opt_stds, opt_colors = ['SGD'], [np.mean(sgd_accs)], [np.std(sgd_accs, ddof=1)], ['#D85A30']

if os.path.exists('adamw_multiseed_results.csv'):
    df_adamw = pd.read_csv('adamw_multiseed_results.csv')
    opt_names.insert(0, 'AdamW')
    opt_means.insert(0, df_adamw['acc'].mean())
    opt_stds.insert(0, df_adamw['acc'].std(ddof=1))
    opt_colors.insert(0, '#7F77DD')

bars = ax.bar(opt_names, opt_means, yerr=opt_stds, color=opt_colors,
              capsize=8, edgecolor='white', linewidth=1.5, alpha=0.88, width=0.5)
for bar, m, s in zip(bars, opt_means, opt_stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.3,
            f'{m:.2f}%\n±{s:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, max(opt_means) + max(opt_stds) + 8)
ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('mean ± std (n=5 seeds)')
ax.axhline(50, color='gray', linestyle=':', linewidth=1, label='chance (50%)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Plot 2 : évolution par seed
ax2 = axes[1]
ax2.plot(range(1, len(SEEDS)+1), sgd_accs, 'o-', color='#D85A30',
         linewidth=2, markersize=8, label='SGD')
ax2.axhline(np.mean(sgd_accs), color='#D85A30', linestyle='--', alpha=0.6,
            label=f'SGD mean={np.mean(sgd_accs):.2f}%')

if os.path.exists('adamw_multiseed_results.csv'):
    df_a = pd.read_csv('adamw_multiseed_results.csv')
    ax2.plot(range(1, len(SEEDS)+1), df_a['acc'].tolist(), 's-', color='#7F77DD',
             linewidth=2, markersize=8, label='AdamW')
    ax2.axhline(df_a['acc'].mean(), color='#7F77DD', linestyle='--', alpha=0.6,
                label=f'AdamW mean={df_a["acc"].mean():.2f}%')

ax2.set_xticks(range(1, len(SEEDS)+1))
ax2.set_xticklabels([f'seed\n{s}' for s in SEEDS])
ax2.set_ylabel('Validation Accuracy (%)')
ax2.set_title('Accuracy par seed')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_adamw_sgd_multiseed.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : comparison_adamw_sgd_multiseed.png')

In [ ]:
# Cell 9 — Résumé final
print('='*60)
print('RÉSUMÉ FINAL — SGD vs AdamW · BERT · SST-2 · Multi-Seeds')
print('='*60)
print(f'  SGD   : {np.mean(sgd_accs):.2f}% ± {np.std(sgd_accs, ddof=1):.2f}% (IC95: [{bootstrap_ci(sgd_accs)[0]:.2f}, {bootstrap_ci(sgd_accs)[1]:.2f}])')

if os.path.exists('adamw_multiseed_results.csv'):
    adamw_accs = pd.read_csv('adamw_multiseed_results.csv')['acc'].tolist()
    ci_a = bootstrap_ci(adamw_accs)
    t_stat, p_val = stats.ttest_ind(adamw_accs, sgd_accs, equal_var=False)
    d = (np.mean(adamw_accs)-np.mean(sgd_accs))/np.sqrt((np.std(adamw_accs,ddof=1)**2+np.std(sgd_accs,ddof=1)**2)/2)
    print(f'  AdamW : {np.mean(adamw_accs):.2f}% ± {np.std(adamw_accs, ddof=1):.2f}% (IC95: [{ci_a[0]:.2f}, {ci_a[1]:.2f}])')
    print()
    print(f'  Test de Welch : t={t_stat:.3f}, p={p_val:.6f}')
    print(f'  Cohen\'s d     : {d:.2f} (effet {"large" if abs(d)>0.8 else "moyen" if abs(d)>0.5 else "petit"})')
    print()
    print('  CONCLUSION :')
    print(f'  AdamW surpasse SGD de {np.mean(adamw_accs)-np.mean(sgd_accs):.1f}% en accuracy.')
    sig = 'p < 0.001 (hautement significatif)' if p_val < 0.001 else ('p < 0.01' if p_val < 0.01 else 'p < 0.05')
    print(f'  Cette différence est statistiquement significative ({sig})')
    print(f'  sur {len(SEEDS)} seeds indépendantes — le résultat n\'est pas dû au hasard.')